# Baixar Assets de Stock para Auto Video Producer

Este notebook baixa vídeos e músicas de bancos gratuitos (Pexels, Pixabay, Mixkit) direto para o Google Drive,
organizados nas pastas que o pipeline espera.

## 1. Montar Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Definir pasta de destino

Edite `PASTA_DESTINO` se quiser outro caminho no Drive.

In [2]:
import os
PASTA_DESTINO = "/content/drive/MyDrive/auto-video-producer/assets"
os.makedirs(PASTA_DESTINO, exist_ok=True)
print(f"Assets serão salvos em: {PASTA_DESTINO}")

## 3. Instalar dependências

In [3]:
!pip install -q requests tqdm

## 4. Baixar vídeos do Pexels (exemplo)

Você precisa de uma API key gratuita do Pexels: https://www.pexels.com/api/

In [4]:
import requests
from tqdm import tqdm
import time

PEXELS_API_KEY = "SUA_CHAVE_PEXELS_AQUI"  # <- COLOQUE SUA CHAVE

def baixar_pexels(query, categoria, max_videos=5, orientation="portrait"):
    url = "https://api.pexels.com/videos/search"
    headers = {"Authorization": PEXELS_API_KEY}
    params = {"query": query, "per_page": max_videos, "orientation": orientation}
    
    resp = requests.get(url, headers=headers, params=params)
    if resp.status_code != 200:
        print(f"Erro {resp.status_code}: {resp.text}")
        return
    
    dados = resp.json()
    pasta_cat = os.path.join(PASTA_DESTINO, "videos", categoria)
    os.makedirs(pasta_cat, exist_ok=True)
    
    for video in dados.get("videos", []):
        # Pega o arquivo de maior qualidade
        arquivos = sorted(video["video_files"], key=lambda x: x.get("width", 0), reverse=True)
        if not arquivos:
            continue
        melhor = arquivos[0]
        video_url = melhor["link"]
        
        nome = f"pexels_{video['id']}.mp4"
        caminho = os.path.join(pasta_cat, nome)
        
        if os.path.exists(caminho):
            print(f"  Já existe: {nome}")
            continue
        
        print(f"Baixando: {nome} ({melhor.get('width','?')}x{melhor.get('height','?')})")
        r = requests.get(video_url, stream=True)
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(caminho, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as pbar:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))
        time.sleep(0.5)  # rate limit

# Exemplos de categorias e buscas
buscas = [
    ("nature forest", "natureza"),
    ("ocean waves", "agua"),
    ("mountain landscape", "natureza"),
    ("city timelapse", "cidade"),
    ("technology abstract", "tecnologia"),
    ("people working", "pessoas"),
    ("sky clouds", "ceu"),
    ("office workspace", "escritorio"),
    ("home interior", "casa"),
]

for query, cat in buscas:
    print(f"\n=== {cat}: {query} ===")
    baixar_pexels(query, cat, max_videos=3)

## 5. Baixar músicas do Mixkit (gratuito, sem API key)

Mixkit oferece músicas royalty-free: https://mixkit.co/free-stock-music/

In [5]:
# Mixkit não tem API pública fácil, mas você pode baixar manualmente
# e colocar em assets/musicas/
musicas_dir = os.path.join(PASTA_DESTINO, "musicas")
os.makedirs(musicas_dir, exist_ok=True)
print(f"Coloque suas músicas (.mp3/.wav) em: {musicas_dir}")
print("Sugestão: baixe de https://mixkit.co/free-stock-music/ ou https://pixabay.com/music/")

## 6. Verificar estrutura final

In [6]:
import os
for root, dirs, files in os.walk(PASTA_DESTINO):
    level = root.replace(PASTA_DESTINO, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = '  ' * (level + 1)
    for f in files[:5]:  # mostra só 5 primeiros
        print(f"{subindent}{f}")
    if len(files) > 5:
        print(f"{subindent}... e mais {len(files)-5} arquivo(s)")